In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

sc.settings.verbosity = 0

In [2]:
import GenKI as gk
from GenKI.preprocesing import build_adata
from GenKI.dataLoader import DataLoader
from GenKI.train import VGAE_trainer
from GenKI import utils

%load_ext autoreload
%autoreload 2

/Users/siam/anaconda3/envs/genki/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/Users/siam/anaconda3/envs/genki/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/Users/siam/anaconda3/envs/genki/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/Users/siam/anaconda3/envs/genki/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  return module_get

In [ ]:
adata = sc.read_h5ad('adata.h5ad')
adata

AnnData object with n_obs × n_vars = 3091 × 5000
    obs: 'Sample', 'Group', 'Batch', 'sample', 'n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'n_genes', '_scvi_batch', '_scvi_labels', 'doublet', 'doublet_score', 'singlet_score', 'leiden', 'size_factor', '_indices', '_scvi_sample', 'NTM_DE_eff_size', 'Symptomatic_DA_lfc', 'cell_subtype', 'batch', 'cell_type', 'Age'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'Group_colors', '_scvi_manager_uuid', '_scvi_uuid', 'cell_type_colors', 'clinical_data', 'leiden', 'leiden_colors', 'neighbors', 'pca', 'rank_genes_groups', 'umap'
    obsm: 'X_mrVI', 'X_pca', 'X_pic

In [4]:
# Select top 1000 highly variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=1000, flavor='seurat_v3')
adata = adata[:, adata.var['highly_variable']].copy()

print(f"After HVG selection: {adata.n_obs} cells × {adata.n_vars} genes")
adata

/var/folders/ry/99nx7wld5jjcxhnlkbt0651r0000gn/T/ipykernel_16035/2992170871.py:2: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  sc.pp.highly_variable_genes(adata, n_top_genes=1000, flavor='seurat_v3')


After HVG selection: 3091 cells × 1000 genes


AnnData object with n_obs × n_vars = 3091 × 1000
    obs: 'Sample', 'Group', 'Batch', 'sample', 'n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'n_genes', '_scvi_batch', '_scvi_labels', 'doublet', 'doublet_score', 'singlet_score', 'leiden', 'size_factor', '_indices', '_scvi_sample', 'NTM_DE_eff_size', 'Symptomatic_DA_lfc', 'cell_subtype', 'batch', 'cell_type', 'Age'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'Group_colors', '_scvi_manager_uuid', '_scvi_uuid', 'cell_type_colors', 'clinical_data', 'leiden', 'leiden_colors', 'neighbors', 'pca', 'rank_genes_groups', 'umap', 'hvg'
    obsm: 'X_mrVI', 'X_pca',

In [5]:
from GenKI.preprocesing import build_adata

# Preprocess your raw adata
adata = build_adata(
    adata,                    # Your raw AnnData object
    log_normalize=True,       # Optional: log-normalize counts
    scale_data=True,          # Required: creates adata.layers["norm"] and standardizes adata.X
    as_sparse=True,           # Optional: store as sparse matrix
    uppercase=True,           # Optional: convert gene names to uppercase
)

In [19]:
# Search for FOS in var names
ko = 'FOS'

gene_indices = np.where(adata.var_names.str.contains(ko))[0]
print(f"{ko} found at indices: {gene_indices}")

# If found, display the gene names
if len(gene_indices) > 0:
    print(f"{ko} gene names:")
    for idx in gene_indices:
        print(f"Index {idx}: {adata.var_names[idx]}")
else:
    print(f"{ko} not found in var names")

FOS found at indices: [295 296 297]
FOS gene names:
Index 295: FOS
Index 296: FOSB
Index 297: FOSL1


In [8]:
# gene to ko
adata.var_names[295]

'FOS'

In [9]:
# load data

data_wrapper =  DataLoader(
                adata, # adata object
                target_gene = [295], # KO gene name/index
                target_cell = None, # obsname for cell type, if none use all
                obs_label = "cell_type", # colname for genes
                GRN_file_dir = "GRNs", # folder name for GRNs
                rebuild_GRN = True, # whether build GRN by pcNet
                pcNet_name = "pcNet_example", # GRN file name
                verbose = True, # whether verbose
                n_cpus = 12, # multiprocessing
                )

data_wt = data_wrapper.load_data()
data_ko = data_wrapper.load_kodata()

use all the cells (3091) in adata
build GRN


2026-03-13 09:18:14,409	INFO worker.py:2013 -- Started a local Ray instance.
/Users/siam/anaconda3/envs/genki/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


ray init, using 12 CPUs
execution time of making pcNet: 767.66 s
GRN has been built and saved in "GRNs/pcNet_example.npz"
init completed



In [10]:
# init trainer

hyperparams = {"epochs": 100, 
               "lr": 7e-4, 
               "beta": 1e-4, 
               "seed": 8096}
log_dir=None 

sensei = VGAE_trainer(data_wt, 
                     epochs=hyperparams["epochs"], 
                     lr=hyperparams["lr"], 
                     log_dir=log_dir, 
                     beta=hyperparams["beta"],
                     seed=hyperparams["seed"],
                     verbose=True,
                     )

In [11]:
# %%timeit
sensei.train()

Epoch: 000, Loss: 1.8038, AUROC: 0.9297, AP: 0.9365
Epoch: 001, Loss: 1.7513, AUROC: 0.9558, AP: 0.9527
Epoch: 002, Loss: 1.6875, AUROC: 0.9546, AP: 0.9511
Epoch: 003, Loss: 1.7128, AUROC: 0.9522, AP: 0.9488
Epoch: 004, Loss: 1.6913, AUROC: 0.9511, AP: 0.9477
Epoch: 005, Loss: 1.7003, AUROC: 0.9507, AP: 0.9473
Epoch: 006, Loss: 1.6549, AUROC: 0.9505, AP: 0.9472
Epoch: 007, Loss: 1.6403, AUROC: 0.9506, AP: 0.9473
Epoch: 008, Loss: 1.6228, AUROC: 0.9510, AP: 0.9476
Epoch: 009, Loss: 1.5738, AUROC: 0.9515, AP: 0.9481
Epoch: 010, Loss: 1.5951, AUROC: 0.9523, AP: 0.9488
Epoch: 011, Loss: 1.5531, AUROC: 0.9531, AP: 0.9495
Epoch: 012, Loss: 1.5376, AUROC: 0.9540, AP: 0.9503
Epoch: 013, Loss: 1.5835, AUROC: 0.9549, AP: 0.9511
Epoch: 014, Loss: 1.5754, AUROC: 0.9559, AP: 0.9521
Epoch: 015, Loss: 1.5319, AUROC: 0.9571, AP: 0.9530
Epoch: 016, Loss: 1.5204, AUROC: 0.9582, AP: 0.9540
Epoch: 017, Loss: 1.5157, AUROC: 0.9592, AP: 0.9549
Epoch: 018, Loss: 1.5359, AUROC: 0.9602, AP: 0.9558
Epoch: 019, 

In [ ]:
sensei.save_model('model')

save model parameters to model/model_ntm_neutrophils.th


In [13]:
# get distance between wt and ko

z_mu_wt, z_std_wt = sensei.get_latent_vars(data_wt)
z_mu_ko, z_std_ko = sensei.get_latent_vars(data_ko)
dis = gk.utils.get_distance(z_mu_ko, z_std_ko, z_mu_wt, z_std_wt, by="KL")

In [14]:
# raw ranked gene list

res_raw = utils.get_generank(data_wt, dis, rank=True)
res_raw.head()

,dis,rank
FOS,222.885655,1
ACAP1,0.008172,2
DIAPH1,0.007997,3
KLF2,0.007582,4
VIM,0.007180,5


In [15]:
# if permutation test

null = sensei.pmt(data_ko, n=100, by="KL")
res = utils.get_generank(data_wt, dis, null,)
#                       save_significant_as = 'gene_list_example')
res

Permutating: 100%|██████████| 100/100 [00:04<00:00, 23.75it/s]


,dis,index,hit,rank
FOS,222.885655,295,100,1
ACAP1,0.008172,5,100,2
DIAPH1,0.007997,213,100,3
KLF2,0.007582,437,100,4
VIM,0.007180,946,100,5
MYADM,0.007093,541,100,6
MCL1,0.006991,503,100,7
IRF1,0.006958,406,100,8
PRKACA,0.006861,671,99,9
LAPTM5,0.006846,455,100,10


In [16]:
res = utils.get_generank(data_wt, dis, null, cutoff=0)  # Include all genes with hits

In [17]:
res.to_csv("res.csv", index=True)

In [18]:
res

,dis,index,hit,rank
FOS,222.885655,295,100,1
ACAP1,0.008172,5,100,2
DIAPH1,0.007997,213,100,3
KLF2,0.007582,437,100,4
VIM,0.007180,946,100,5
...,...,...,...,...
MUC19,0.000032,540,1,112
COPZ2,0.000031,177,1,113
ZNF429,0.000030,982,2,114
EID2B,0.000030,236,1,115
